In [ ]:
# Supreme Court Judgement IR System with PDF Download - Complete Implementation
# Run this single cell to launch the web application

import sys
import os
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from collections import defaultdict, Counter
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import glob
import time
import heapq
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Try to import Flask, install if not available
try:
    from flask import Flask, render_template_string, request, jsonify, send_file, Response
    from werkzeug.serving import make_server
    import threading
    from io import BytesIO
    import mimetypes
    FLASK_AVAILABLE = True
except ImportError:
    print("Installing required packages...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flask", "werkzeug", "scikit-learn", "nltk", "numpy", "pandas"])
    from flask import Flask, render_template_string, request, jsonify, send_file, Response
    from werkzeug.serving import make_server
    import threading
    from io import BytesIO
    import mimetypes
    FLASK_AVAILABLE = True

print("✓ All packages installed and ready!")

# ========================= CONFIGURATION =========================
# Update these paths to match your directory structure
PDF_DIR = r"C:\Users\Armaghan Rafique\Desktop\AI Project\supreme_court_judgements_pdf"
TXT_DIR = r"C:\Users\Armaghan Rafique\Desktop\AI Project\supreme_court_judgements_txt"
INDEX_DIR = "indices"

# Create necessary directories
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

print(f"PDF Directory: {PDF_DIR}")
print(f"TXT Directory: {TXT_DIR}")
print(f"Index Directory: {INDEX_DIR}")

# ========================= PDF to TEXT CONVERTER =========================
def convert_all_pdfs_to_text():
    """Convert ALL PDF files to text files"""
    # Check if TXT directory already has files
    txt_files = list(glob.glob(os.path.join(TXT_DIR, "*.txt")))
    
    if len(txt_files) > 0:
        print(f"Found {len(txt_files)} existing text files in {TXT_DIR}")
        return True
    
    try:
        import PyPDF2
    except ImportError:
        print("Installing PyPDF2 for PDF conversion...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])
        import PyPDF2
    
    pdf_files = list(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
    
    if len(pdf_files) == 0:
        print(f"No PDF files found in {PDF_DIR}")
        print("Please ensure your PDF files are in the correct directory.")
        return False
    
    print(f"Found {len(pdf_files)} PDF files. Converting ALL to text...")
    print("This may take a while depending on the number of PDFs...")
    
    converted_count = 0
    failed_count = 0
    total_files = len(pdf_files)
    
    for i, pdf_file in enumerate(pdf_files):
        try:
            txt_filename = os.path.basename(pdf_file).replace('.pdf', '.txt')
            txt_path = os.path.join(TXT_DIR, txt_filename)
            
            with open(pdf_file, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                
                # Only save if we extracted some text
                if text.strip():
                    with open(txt_path, 'w', encoding='utf-8') as txt_file:
                        txt_file.write(text)
                    converted_count += 1
                else:
                    print(f"  ⚠ No text extracted from {os.path.basename(pdf_file)}")
                    failed_count += 1
            
            # Progress update
            if (i + 1) % 50 == 0 or (i + 1) == total_files:
                print(f"  Progress: {i + 1}/{total_files} files ({converted_count} converted, {failed_count} failed)")
            
        except Exception as e:
            failed_count += 1
            print(f"✗ Error converting {os.path.basename(pdf_file)}: {e}")
            continue
    
    print(f"\n✓ Conversion complete!")
    print(f"  Total PDFs processed: {total_files}")
    print(f"  Successfully converted: {converted_count}")
    print(f"  Failed conversions: {failed_count}")
    
    return converted_count > 0

# ========================= SEARCH CLASSES =========================
class BooleanRetrieval:
    def __init__(self):
        self.inverted_index = defaultdict(set)
        self.documents = []
        self.doc_ids = []
        self.doc_metadata = {}
        
    def build_index(self, documents, doc_ids, doc_metadata):
        """Build inverted index for Boolean retrieval"""
        self.documents = documents
        self.doc_ids = doc_ids
        self.doc_metadata = doc_metadata
        
        print("Building Boolean index...")
        total_docs = len(documents)
        
        for doc_idx, (doc_id, doc_text) in enumerate(zip(doc_ids, documents)):
            tokens = self.tokenize(doc_text)
            for token in tokens:
                self.inverted_index[token].add(doc_id)
            
            # Progress update
            if (doc_idx + 1) % 100 == 0 or (doc_idx + 1) == total_docs:
                print(f"  Processed {doc_idx + 1}/{total_docs} documents...")
        
        print(f"✓ Boolean index built with {len(self.inverted_index)} unique terms")
        
        # Save the index
        index_path = os.path.join(INDEX_DIR, 'boolean_index.pkl')
        with open(index_path, 'wb') as f:
            pickle.dump({
                'inverted_index': self.inverted_index,
                'doc_ids': doc_ids,
                'doc_metadata': doc_metadata
            }, f)
        print(f"  Saved to {index_path}")
    
    def tokenize(self, text):
        """Tokenize and clean text"""
        tokens = word_tokenize(text.lower())
        tokens = [token for token in tokens if token.isalnum() and len(token) > 2]
        stop_words = set(stopwords.words('english'))
        tokens = [token for token in tokens if token not in stop_words]
        return tokens
    
    def search(self, query):
        """Execute Boolean query"""
        query = query.lower().strip()
        
        if ' and not ' in query:
            parts = query.split(' and not ')
            and_result = self._process_and(parts[0])
            not_result = self._process_term(parts[1])
            result_set = and_result - not_result
        elif ' or ' in query:
            parts = query.split(' or ')
            result_set = set()
            for part in parts:
                result_set.update(self._process_term(part.strip()))
        elif ' and ' in query:
            result_set = self._process_and(query)
        else:
            result_set = self._process_term(query)
        
        results = []
        for doc_id in result_set:
            results.append({
                'doc_id': doc_id,
                'score': 1.0,
                'type': 'boolean',
                **self.doc_metadata.get(doc_id, {})
            })
        
        return results
    
    def _process_term(self, term):
        term = term.strip()
        if ' ' in term:
            terms = [t.strip() for t in term.split()]
            if not terms:
                return set()
            term_sets = [self.inverted_index.get(t, set()) for t in terms]
            if not term_sets:
                return set()
            result = term_sets[0]
            for term_set in term_sets[1:]:
                result = result.intersection(term_set)
            return result
        elif '*' in term:
            pattern = term.replace('*', '.*')
            regex = re.compile(f'^{pattern}$')
            matching_terms = [t for t in self.inverted_index.keys() if regex.match(t)]
            result = set()
            for t in matching_terms:
                result.update(self.inverted_index.get(t, set()))
            return result
        else:
            return self.inverted_index.get(term, set())
    
    def _process_and(self, query):
        terms = [t.strip() for t in query.split(' and ')]
        terms = [t for t in terms if t]
        if not terms:
            return set()
        result = self._process_term(terms[0])
        for term in terms[1:]:
            result = result.intersection(self._process_term(term))
        return result

class TFIDFRetrieval:
    def __init__(self, max_features=10000):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        )
        self.tfidf_matrix = None
        self.doc_ids = []
        self.feature_names = []
        self.doc_metadata = {}
        
    def build_index(self, documents, doc_ids, doc_metadata):
        self.doc_ids = doc_ids
        self.doc_metadata = doc_metadata
        
        print("Building TF-IDF index...")
        print(f"  Processing {len(documents)} documents...")
        
        self.tfidf_matrix = self.vectorizer.fit_transform(documents)
        self.feature_names = self.vectorizer.get_feature_names_out()
        
        print(f"✓ TF-IDF index built with {len(self.feature_names)} features")
        print(f"  Document-term matrix shape: {self.tfidf_matrix.shape}")
        
        # Save the index
        index_path = os.path.join(INDEX_DIR, 'tfidf_index.pkl')
        with open(index_path, 'wb') as f:
            pickle.dump({
                'vectorizer': self.vectorizer,
                'tfidf_matrix': self.tfidf_matrix,
                'doc_ids': doc_ids,
                'doc_metadata': doc_metadata
            }, f)
        print(f"  Saved to {index_path}")
    
    def search(self, query, top_k=10):
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.tfidf_matrix).flatten()
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            if similarities[idx] > 0.01:
                doc_id = self.doc_ids[idx]
                results.append({
                    'doc_id': doc_id,
                    'score': float(similarities[idx]),
                    'type': 'semantic',
                    **self.doc_metadata.get(doc_id, {})
                })
        
        return results
    
    def get_similar_documents(self, doc_id, top_k=5):
        if doc_id not in self.doc_ids:
            return []
        doc_idx = self.doc_ids.index(doc_id)
        doc_vector = self.tfidf_matrix[doc_idx:doc_idx+1]
        similarities = cosine_similarity(doc_vector, self.tfidf_matrix).flatten()
        similarities[doc_idx] = -1
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            if similarities[idx] > 0.01:
                similar_doc_id = self.doc_ids[idx]
                results.append({
                    'doc_id': similar_doc_id,
                    'score': float(similarities[idx]),
                    **self.doc_metadata.get(similar_doc_id, {})
                })
        
        return results

class NGramRetrieval:
    def __init__(self, n=2):
        self.n = n
        self.ngram_index = defaultdict(set)
        self.documents = []
        self.doc_ids = []
        self.doc_metadata = {}
        
    def build_index(self, documents, doc_ids, doc_metadata):
        self.documents = documents
        self.doc_ids = doc_ids
        self.doc_metadata = doc_metadata
        
        print(f"Building {self.n}-gram index...")
        total_docs = len(documents)
        
        for doc_idx, (doc_id, doc_text) in enumerate(zip(doc_ids, documents)):
            words = doc_text.lower().split()
            for i in range(len(words) - self.n + 1):
                ngram = ' '.join(words[i:i+self.n])
                self.ngram_index[ngram].add(doc_id)
            
            # Progress update
            if (doc_idx + 1) % 100 == 0 or (doc_idx + 1) == total_docs:
                print(f"  Processed {doc_idx + 1}/{total_docs} documents...")
        
        print(f"✓ {self.n}-gram index built with {len(self.ngram_index)} n-grams")
        
        # Save the index
        index_path = os.path.join(INDEX_DIR, f'ngram_{self.n}_index.pkl')
        with open(index_path, 'wb') as f:
            pickle.dump({
                'ngram_index': self.ngram_index,
                'doc_ids': doc_ids,
                'doc_metadata': doc_metadata
            }, f)
        print(f"  Saved to {index_path}")
    
    def search(self, phrase, top_k=10):
        phrase = phrase.lower().strip()
        words = phrase.split()
        if len(words) != self.n:
            return []
        
        matching_docs = self.ngram_index.get(phrase, set())
        results = []
        for doc_id in list(matching_docs)[:top_k]:
            results.append({
                'doc_id': doc_id,
                'score': self.n * 0.2,
                'type': f'phrase_{self.n}gram',
                **self.doc_metadata.get(doc_id, {})
            })
        
        return results

# ========================= MAIN IR SYSTEM =========================
class IRSystem:
    def __init__(self, use_existing_indices=True):
        self.boolean_retrieval = BooleanRetrieval()
        self.tfidf_retrieval = TFIDFRetrieval(max_features=15000)
        self.ngram_retrievals = {
            2: NGramRetrieval(n=2),
            3: NGramRetrieval(n=3)
        }
        
        self.documents = []
        self.doc_ids = []
        self.doc_metadata = {}
        self.total_docs = 0
        
        # Map document IDs to PDF file paths
        self.pdf_paths = {}
        
        self.load_all_data()
        
        if use_existing_indices and self.indices_exist():
            print("\nChecking if existing indices are up to date...")
            if self.indices_need_rebuild():
                print("Indices are outdated or missing. Rebuilding...")
                self.build_indices()
            else:
                self.load_indices()
        else:
            self.build_indices()
    
    def load_all_data(self):
        """Load ALL text files from the TXT directory"""
        print("\nLoading documents from text files...")
        
        # First check if we need to convert PDFs
        txt_files = glob.glob(os.path.join(TXT_DIR, "*.txt"))
        if not txt_files:
            print("No text files found. Checking for PDF files...")
            if not convert_all_pdfs_to_text():
                return
            txt_files = glob.glob(os.path.join(TXT_DIR, "*.txt"))
        
        print(f"Found {len(txt_files)} text files in {TXT_DIR}")
        
        # Also get PDF files for download mapping
        pdf_files = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
        pdf_mapping = {os.path.basename(p).replace('.pdf', ''): p for p in pdf_files}
        
        loaded_count = 0
        skipped_count = 0
        total_files = len(txt_files)
        
        print(f"Processing ALL {total_files} text files...")
        
        for i, txt_file in enumerate(txt_files):
            try:
                doc_id = os.path.basename(txt_file).replace('.txt', '')
                
                # Get file size
                file_size = os.path.getsize(txt_file)
                if file_size == 0:
                    skipped_count += 1
                    continue
                
                with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                
                # Check if content is meaningful
                if len(content.strip()) > 50:
                    self.documents.append(content)
                    self.doc_ids.append(doc_id)
                    
                    # Store PDF path if available
                    if doc_id in pdf_mapping:
                        self.pdf_paths[doc_id] = pdf_mapping[doc_id]
                    
                    # Extract some metadata
                    words = content.split()
                    sentences = sent_tokenize(content[:5000])
                    
                    self.doc_metadata[doc_id] = {
                        'filename': os.path.basename(txt_file),
                        'pdf_filename': f"{doc_id}.pdf" if doc_id in pdf_mapping else None,
                        'has_pdf': doc_id in pdf_mapping,
                        'title': doc_id.replace('_', ' ').title(),
                        'length': len(content),
                        'word_count': len(words),
                        'sentence_count': len(sentences),
                        'file_size_kb': round(file_size / 1024, 2)
                    }
                    
                    loaded_count += 1
                else:
                    skipped_count += 1
                
                # Progress update
                if (i + 1) % 100 == 0 or (i + 1) == total_files:
                    print(f"  Loaded {i + 1}/{total_files} files ({loaded_count} valid, {skipped_count} skipped)")
                
            except Exception as e:
                skipped_count += 1
                print(f"  Error loading {txt_file}: {e}")
                continue
        
        self.total_docs = len(self.documents)
        print(f"\n✓ Successfully loaded {self.total_docs} documents")
        print(f"✓ PDF files available for download: {len(self.pdf_paths)} documents")
        
        if self.total_docs == 0:
            print("\n⚠ WARNING: No documents were loaded!")
            print("Possible issues:")
            print("1. Check that your TXT_DIR contains .txt files")
            print("2. Files might be empty or too small")
            print("3. Check file permissions")
    
    def indices_exist(self):
        """Check if all index files exist"""
        required_files = [
            'boolean_index.pkl',
            'tfidf_index.pkl',
            'ngram_2_index.pkl'
        ]
        for file in required_files:
            if not os.path.exists(os.path.join(INDEX_DIR, file)):
                print(f"  Missing index file: {file}")
                return False
        return True
    
    def indices_need_rebuild(self):
        """Check if indices need to be rebuilt"""
        if not self.indices_exist():
            return True
        
        try:
            with open(os.path.join(INDEX_DIR, 'boolean_index.pkl'), 'rb') as f:
                data = pickle.load(f)
                indexed_doc_count = len(data['doc_ids'])
                if indexed_doc_count != self.total_docs:
                    print(f"  Document count changed: {indexed_doc_count} -> {self.total_docs}")
                    return True
        except:
            return True
        
        return False
    
    def build_indices(self):
        """Build all search indices from scratch"""
        print("\n" + "="*60)
        print(f"Building Search Indices for {self.total_docs} documents")
        print("="*60)
        
        if not self.documents:
            print("No documents to index!")
            return
        
        start_time = time.time()
        
        # Build Boolean index
        self.boolean_retrieval.build_index(self.documents, self.doc_ids, self.doc_metadata)
        
        # Build TF-IDF index
        self.tfidf_retrieval.build_index(self.documents, self.doc_ids, self.doc_metadata)
        
        # Build N-gram indices (only 2 for large collections)
        self.ngram_retrievals[2].build_index(self.documents, self.doc_ids, self.doc_metadata)
        
        elapsed_time = time.time() - start_time
        print(f"\n✓ All indices built in {elapsed_time:.2f} seconds")
        
        # Print index statistics
        print("\nIndex Statistics:")
        print(f"- Boolean index terms: {len(self.boolean_retrieval.inverted_index)}")
        print(f"- TF-IDF features: {len(self.tfidf_retrieval.feature_names)}")
        print(f"- 2-gram index: {len(self.ngram_retrievals[2].ngram_index)} n-grams")
    
    def load_indices(self):
        """Load existing indices from disk"""
        print("Loading existing indices...")
        
        indices_loaded = 0
        total_indices = 3
        
        try:
            # Load Boolean index
            print("  Loading Boolean index...")
            with open(os.path.join(INDEX_DIR, 'boolean_index.pkl'), 'rb') as f:
                data = pickle.load(f)
                self.boolean_retrieval.inverted_index = data['inverted_index']
                self.boolean_retrieval.doc_ids = data['doc_ids']
                self.boolean_retrieval.doc_metadata = data['doc_metadata']
                self.doc_metadata.update(data['doc_metadata'])
            indices_loaded += 1
            
            # Load TF-IDF index
            print("  Loading TF-IDF index...")
            with open(os.path.join(INDEX_DIR, 'tfidf_index.pkl'), 'rb') as f:
                data = pickle.load(f)
                self.tfidf_retrieval.vectorizer = data['vectorizer']
                self.tfidf_retrieval.tfidf_matrix = data['tfidf_matrix']
                self.tfidf_retrieval.doc_ids = data['doc_ids']
                self.tfidf_retrieval.doc_metadata = data['doc_metadata']
            indices_loaded += 1
            
            # Load 2-gram index
            index_file = os.path.join(INDEX_DIR, 'ngram_2_index.pkl')
            if os.path.exists(index_file):
                print("  Loading 2-gram index...")
                try:
                    with open(index_file, 'rb') as f:
                        data = pickle.load(f)
                        self.ngram_retrievals[2].ngram_index = data['ngram_index']
                        self.ngram_retrievals[2].doc_ids = data['doc_ids']
                        self.ngram_retrievals[2].doc_metadata = data['doc_metadata']
                    indices_loaded += 1
                except Exception as e:
                    print(f"    Warning: Could not load 2-gram index: {e}")
            
            print(f"\n✓ Successfully loaded {indices_loaded}/{total_indices} indices")
            
        except Exception as e:
            print(f"\n✗ Error loading indices: {e}")
            print("Rebuilding indices...")
            self.build_indices()
    
    def get_pdf_path(self, doc_id):
        """Get PDF file path for a document ID"""
        return self.pdf_paths.get(doc_id)
    
    def hybrid_search(self, query, search_type='boolean', top_k=10):
        """Perform search using specified search type"""
        start_time = time.time()
        
        if search_type == 'boolean':
            results = self.boolean_retrieval.search(query)
        elif search_type == 'semantic':
            results = self.tfidf_retrieval.search(query, top_k=top_k)
        elif search_type == 'phrase':
            all_results = []
            words = query.lower().split()
            if len(words) >= 2:
                results = self.ngram_retrievals[2].search(' '.join(words[:2]), top_k=top_k)
            else:
                results = []
        elif search_type == 'wildcard':
            results = self.boolean_retrieval.search(query)
        else:
            results = []
        
        elapsed_time = time.time() - start_time
        
        # Add PDF availability info to results
        for result in results:
            doc_id = result['doc_id']
            result['has_pdf'] = doc_id in self.pdf_paths
            result['pdf_filename'] = f"{doc_id}.pdf" if doc_id in self.pdf_paths else None
        
        return {
            'results': results[:top_k],
            'query_time': elapsed_time,
            'total_results': len(results),
            'query': query,
            'type': search_type
        }
    
    def get_document_content(self, doc_id, max_chars=5000):
        """Get document content with preview"""
        if doc_id in self.doc_ids:
            idx = self.doc_ids.index(doc_id)
            content = self.documents[idx]
            if len(content) > max_chars:
                # Try to find a good break point
                preview = content[:max_chars]
                last_period = preview.rfind('.')
                if last_period > max_chars * 0.8:
                    preview = preview[:last_period + 1]
                return preview + '...'
            return content
        return None
    
    def get_similar_documents(self, doc_id, top_k=5):
        """Get similar documents using TF-IDF"""
        return self.tfidf_retrieval.get_similar_documents(doc_id, top_k)
    
    def get_document_stats(self):
        """Get system statistics"""
        if not self.documents:
            return {}
        
        total_words = sum(len(doc.split()) for doc in self.documents)
        avg_words = total_words / self.total_docs if self.total_docs > 0 else 0
        
        # Sample some documents for term frequency
        sample_size = min(100, self.total_docs)
        if sample_size > 0:
            sample_indices = np.random.choice(self.total_docs, sample_size, replace=False)
            
            all_terms = []
            for idx in sample_indices:
                doc = self.documents[idx]
                tokens = word_tokenize(doc.lower())
                tokens = [t for t in tokens if t.isalnum() and len(t) > 3]
                all_terms.extend(tokens[:100])
            
            term_freq = Counter(all_terms)
            top_terms = term_freq.most_common(12)
        else:
            top_terms = []
        
        # Calculate average file size
        total_size_kb = sum(meta.get('file_size_kb', 0) for meta in self.doc_metadata.values())
        avg_size_kb = total_size_kb / self.total_docs if self.total_docs > 0 else 0
        
        return {
            'total_documents': self.total_docs,
            'total_words': total_words,
            'average_words_per_document': round(avg_words, 2),
            'average_file_size_kb': round(avg_size_kb, 2),
            'indexed_terms': len(self.boolean_retrieval.inverted_index) if hasattr(self.boolean_retrieval, 'inverted_index') else 0,
            'pdf_available': len(self.pdf_paths),
            'top_terms': [{'term': term, 'count': count} for term, count in top_terms]
        }

# ========================= HTML TEMPLATE =========================
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Supreme Court Judgement IR System</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/css/bootstrap.min.css" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css">
    <style>
        body {
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
            min-height: 100vh;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        .navbar {
            background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
            box-shadow: 0 4px 20px rgba(0,0,0,0.15);
        }
        .main-container {
            margin-top: 20px;
            animation: fadeIn 0.8s ease-in;
        }
        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }
        .search-card {
            background: white;
            border-radius: 15px;
            padding: 30px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.1);
            border: none;
            margin-bottom: 30px;
        }
        .result-card {
            transition: all 0.3s ease;
            border: none;
            border-left: 5px solid #2a5298;
            margin-bottom: 15px;
            box-shadow: 0 3px 10px rgba(0,0,0,0.08);
        }
        .result-card:hover {
            transform: translateY(-5px);
            box-shadow: 0 8px 25px rgba(0,0,0,0.15);
        }
        .stats-card {
            background: white;
            border-radius: 10px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 5px 15px rgba(0,0,0,0.08);
            text-align: center;
        }
        .tab-content {
            padding-top: 20px;
        }
        .example-query {
            cursor: pointer;
            padding: 8px 12px;
            margin: 5px;
            background: #f8f9fa;
            border-radius: 5px;
            border: 1px solid #e9ecef;
            display: inline-block;
            transition: all 0.2s;
        }
        .example-query:hover {
            background: #e9ecef;
            transform: scale(1.02);
        }
        .badge-search {
            font-size: 0.75em;
            padding: 4px 8px;
        }
        .preview-modal {
            max-height: 70vh;
            overflow-y: auto;
        }
        .btn-download {
            background: linear-gradient(135deg, #28a745 0%, #20c997 100%);
            border: none;
            color: white;
        }
        .btn-download:hover {
            background: linear-gradient(135deg, #218838 0%, #1ba87e 100%);
            color: white;
            transform: scale(1.05);
        }
        .pdf-badge {
            background: linear-gradient(135deg, #dc3545 0%, #c82333 100%);
            color: white;
        }
    </style>
</head>
<body>
    <!-- Navigation -->
    <nav class="navbar navbar-expand-lg navbar-dark">
        <div class="container">
            <a class="navbar-brand" href="#">
                <i class="fas fa-balance-scale fa-2x me-2"></i>
                <div>
                    <h4 class="mb-0">Supreme Court Judgement IR System</h4>
                    <small class="opacity-75">Pakistan Supreme Court Judgements Search Engine</small>
                </div>
            </a>
            <div class="navbar-text">
                <div class="d-flex align-items-center">
                    <div class="me-3">
                        <i class="fas fa-database me-1"></i>
                        <span id="doc-count">{{ stats.total_documents }}</span> Documents
                    </div>
                    <div class="system-status">
                        <span class="badge bg-success">
                            <i class="fas fa-circle me-1"></i>System Ready
                        </span>
                    </div>
                </div>
            </div>
        </div>
    </nav>

    <div class="container main-container">
        <!-- Stats Row -->
        <div class="row mb-4">
            <div class="col-md-3">
                <div class="stats-card">
                    <h5><i class="fas fa-search me-2 text-primary"></i>Search Models</h5>
                    <h3 class="text-primary">4</h3>
                    <small>Boolean, Semantic, Phrase, Wildcard</small>
                </div>
            </div>
            <div class="col-md-3">
                <div class="stats-card">
                    <h5><i class="fas fa-file-pdf me-2 text-danger"></i>PDF Available</h5>
                    <h3 class="text-danger" id="pdf-count">{{ stats.pdf_available }}</h3>
                    <small>Documents with PDF</small>
                </div>
            </div>
            <div class="col-md-3">
                <div class="stats-card">
                    <h5><i class="fas fa-file-word me-2 text-info"></i>Legal Terms</h5>
                    <h3 class="text-info" id="term-count">{{ stats.indexed_terms }}</h3>
                    <small>Indexed Terms</small>
                </div>
            </div>
            <div class="col-md-3">
                <div class="stats-card">
                    <h5><i class="fas fa-chart-bar me-2 text-warning"></i>Avg. Length</h5>
                    <h3 class="text-warning">{{ stats.average_words_per_document }}</h3>
                    <small>Words per Document</small>
                </div>
            </div>
        </div>

        <!-- Search Interface -->
        <div class="search-card">
            <h3 class="mb-4">
                <i class="fas fa-search me-2 text-primary"></i>Search Court Judgements
            </h3>
            
            <!-- Search Tabs -->
            <ul class="nav nav-tabs" id="searchTabs" role="tablist">
                <li class="nav-item" role="presentation">
                    <button class="nav-link active" id="boolean-tab" data-bs-toggle="tab" data-bs-target="#boolean" type="button">
                        <i class="fas fa-code me-1"></i> Boolean Search
                    </button>
                </li>
                <li class="nav-item" role="presentation">
                    <button class="nav-link" id="semantic-tab" data-bs-toggle="tab" data-bs-target="#semantic" type="button">
                        <i class="fas fa-brain me-1"></i> Semantic Search
                    </button>
                </li>
                <li class="nav-item" role="presentation">
                    <button class="nav-link" id="phrase-tab" data-bs-toggle="tab" data-bs-target="#phrase" type="button">
                        <i class="fas fa-quote-right me-1"></i> Phrase Search
                    </button>
                </li>
                <li class="nav-item" role="presentation">
                    <button class="nav-link" id="wildcard-tab" data-bs-toggle="tab" data-bs-target="#wildcard" type="button">
                        <i class="fas fa-asterisk me-1"></i> Wildcard Search
                    </button>
                </li>
            </ul>

            <div class="tab-content p-4 border border-top-0 rounded-bottom" id="searchTabContent">
                <!-- Boolean Search -->
                <div class="tab-pane fade show active" id="boolean" role="tabpanel">
                    <div class="mb-4">
                        <label class="form-label h5">
                            <i class="fas fa-code me-2"></i>Boolean Query Search
                        </label>
                        <div class="input-group input-group-lg">
                            <input type="text" class="form-control" id="booleanQuery" 
                                   placeholder="Enter Boolean query (e.g., murder AND evidence, theft OR robbery)">
                            <button class="btn btn-primary btn-lg" onclick="performSearch('boolean')">
                                <i class="fas fa-search me-1"></i> Search
                            </button>
                        </div>
                        <div class="mt-3">
                            <small class="text-muted">
                                <i class="fas fa-lightbulb me-1"></i>
                                <strong>Boolean Operators:</strong> AND, OR, NOT. 
                                <span class="ms-2">Examples:</span>
                                <span class="example-query" onclick="setExample('boolean', 'murder AND evidence')">murder AND evidence</span>
                                <span class="example-query" onclick="setExample('boolean', 'theft OR robbery')">theft OR robbery</span>
                            </small>
                        </div>
                    </div>
                </div>

                <!-- Semantic Search -->
                <div class="tab-pane fade" id="semantic" role="tabpanel">
                    <div class="mb-4">
                        <label class="form-label h5">
                            <i class="fas fa-brain me-2"></i>Semantic Search
                        </label>
                        <div class="input-group input-group-lg">
                            <input type="text" class="form-control" id="semanticQuery" 
                                   placeholder="Describe what you're looking for (e.g., land dispute resolution)">
                            <button class="btn btn-success btn-lg" onclick="performSearch('semantic')">
                                <i class="fas fa-search me-1"></i> Search
                            </button>
                        </div>
                        <div class="mt-3">
                            <small class="text-muted">
                                <i class="fas fa-lightbulb me-1"></i>
                                Finds conceptually similar documents using TF-IDF and cosine similarity.
                                <span class="ms-2">Examples:</span>
                                <span class="example-query" onclick="setExample('semantic', 'criminal evidence admissibility')">criminal evidence admissibility</span>
                            </small>
                        </div>
                    </div>
                </div>

                <!-- Phrase Search -->
                <div class="tab-pane fade" id="phrase" role="tabpanel">
                    <div class="mb-4">
                        <label class="form-label h5">
                            <i class="fas fa-quote-right me-2"></i>Exact Phrase Search
                        </label>
                        <div class="input-group input-group-lg">
                            <input type="text" class="form-control" id="phraseQuery" 
                                   placeholder="Enter exact phrase (2-3 words, e.g., criminal procedure code)">
                            <button class="btn btn-info btn-lg" onclick="performSearch('phrase')">
                                <i class="fas fa-search me-1"></i> Search
                            </button>
                        </div>
                        <div class="mt-3">
                            <small class="text-muted">
                                <i class="fas fa-lightbulb me-1"></i>
                                Finds exact phrase matches using n-gram technology (2-3 word sequences).
                                <span class="ms-2">Examples:</span>
                                <span class="example-query" onclick="setExample('phrase', 'supreme court')">supreme court</span>
                            </small>
                        </div>
                    </div>
                </div>

                <!-- Wildcard Search -->
                <div class="tab-pane fade" id="wildcard" role="tabpanel">
                    <div class="mb-4">
                        <label class="form-label h5">
                            <i class="fas fa-asterisk me-2"></i>Wildcard Search
                        </label>
                        <div class="input-group input-group-lg">
                            <input type="text" class="form-control" id="wildcardQuery" 
                                   placeholder="Enter pattern with * (e.g., convict*, judg*, viol*)">
                            <button class="btn btn-warning btn-lg" onclick="performSearch('wildcard')">
                                <i class="fas fa-search me-1"></i> Search
                            </button>
                        </div>
                        <div class="mt-3">
                            <small class="text-muted">
                                <i class="fas fa-lightbulb me-1"></i>
                                Use * as wildcard for pattern matching.
                                <span class="ms-2">Examples:</span>
                                <span class="example-query" onclick="setExample('wildcard', 'convict*')">convict* (matches convict, convicted)</span>
                            </small>
                        </div>
                    </div>
                </div>
            </div>

            <!-- Search Options -->
            <div class="row mt-4">
                <div class="col-md-6">
                    <div class="input-group">
                        <span class="input-group-text">
                            <i class="fas fa-list-ol"></i>
                        </span>
                        <select class="form-select" id="resultsPerPage">
                            <option value="5">5 results</option>
                            <option value="10" selected>10 results</option>
                            <option value="20">20 results</option>
                            <option value="50">50 results</option>
                        </select>
                    </div>
                </div>
                <div class="col-md-6 text-end">
                    <div id="performanceStats" class="text-muted">
                        <i class="fas fa-clock me-1"></i>
                        <span id="queryTime">Ready to search</span>
                    </div>
                </div>
            </div>
        </div>

        <!-- Results Section -->
        <div class="search-card" id="resultsSection" style="display: none;">
            <div class="d-flex justify-content-between align-items-center mb-4">
                <h4 class="mb-0">
                    <i class="fas fa-list me-2 text-primary"></i>Search Results
                    <span class="badge bg-primary ms-2" id="resultsCount">0</span>
                    <span class="badge pdf-badge ms-2" id="pdfResultsCount">0 PDFs available</span>
                </h4>
                <button class="btn btn-outline-secondary btn-sm" onclick="clearResults()">
                    <i class="fas fa-times me-1"></i> Clear Results
                </button>
            </div>
            <div id="resultsContainer" class="mt-3">
                <!-- Results will be inserted here -->
            </div>
        </div>

        <!-- Top Terms -->
        <div class="search-card">
            <h5 class="mb-3">
                <i class="fas fa-chart-bar me-2 text-primary"></i>Most Common Legal Terms
            </h5>
            <div class="row">
                {% for term in stats.top_terms %}
                <div class="col-md-3 col-6 mb-2">
                    <div class="d-flex justify-content-between align-items-center p-2 bg-light rounded">
                        <span class="fw-bold">{{ term.term }}</span>
                        <span class="badge bg-primary">{{ term.count }}</span>
                    </div>
                </div>
                {% endfor %}
            </div>
        </div>
    </div>

    <!-- Document Preview Modal -->
    <div class="modal fade" id="documentModal" tabindex="-1">
        <div class="modal-dialog modal-xl">
            <div class="modal-content">
                <div class="modal-header bg-primary text-white">
                    <h5 class="modal-title" id="documentTitle">
                        <i class="fas fa-file-alt me-2"></i>Document Preview
                    </h5>
                    <button type="button" class="btn-close btn-close-white" data-bs-dismiss="modal"></button>
                </div>
                <div class="modal-body preview-modal">
                    <div class="mb-4">
                        <h6 id="documentFilename" class="text-muted mb-2"></h6>
                        <div class="d-flex justify-content-between mb-3">
                            <span class="badge bg-info" id="documentLength"></span>
                            <span class="badge bg-secondary" id="documentWords"></span>
                            <span class="badge bg-success" id="documentSentences"></span>
                        </div>
                    </div>
                    <div class="card">
                        <div class="card-header d-flex justify-content-between align-items-center">
                            <div>
                                <i class="fas fa-align-left me-2"></i>Content Preview
                            </div>
                            <div id="pdfDownloadModal" style="display: none;">
                                <a id="modalDownloadLink" class="btn btn-download btn-sm">
                                    <i class="fas fa-download me-1"></i> Download PDF
                                </a>
                            </div>
                        </div>
                        <div class="card-body">
                            <pre id="documentContent" style="white-space: pre-wrap; font-family: inherit;"></pre>
                        </div>
                    </div>
                    <div class="mt-4">
                        <h6><i class="fas fa-project-diagram me-2"></i>Similar Documents</h6>
                        <div id="similarDocuments" class="mt-2">
                            <div class="text-center py-3">
                                <div class="spinner-border text-primary" role="status">
                                    <span class="visually-hidden">Loading similar documents...</span>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>
                <div class="modal-footer">
                    <button type="button" class="btn btn-secondary" data-bs-dismiss="modal">
                        <i class="fas fa-times me-1"></i> Close
                    </button>
                </div>
            </div>
        </div>
    </div>

    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/js/bootstrap.bundle.min.js"></script>
    <script>
        // Global variables
        let currentSearchType = 'boolean';
        let currentQuery = '';
        
        // Set up tab change listener
        document.addEventListener('DOMContentLoaded', function() {
            const tabEl = document.querySelector('#searchTabs');
            if (tabEl) {
                tabEl.addEventListener('shown.bs.tab', function(event) {
                    const activeTab = event.target.getAttribute('data-bs-target');
                    if (activeTab === '#boolean') currentSearchType = 'boolean';
                    else if (activeTab === '#semantic') currentSearchType = 'semantic';
                    else if (activeTab === '#phrase') currentSearchType = 'phrase';
                    else if (activeTab === '#wildcard') currentSearchType = 'wildcard';
                });
            }
        });

        function setExample(searchType, query) {
            const tabButton = document.querySelector(`[data-bs-target="#${searchType}"]`);
            if (tabButton) {
                const tab = new bootstrap.Tab(tabButton);
                tab.show();
                currentSearchType = searchType;
            }
            
            const inputId = searchType + 'Query';
            const input = document.getElementById(inputId);
            if (input) {
                input.value = query;
                input.focus();
            }
        }

        async function performSearch(searchType = null) {
            if (searchType) {
                currentSearchType = searchType;
            }
            
            const queryInput = document.getElementById(currentSearchType + 'Query');
            currentQuery = queryInput.value.trim();
            
            if (!currentQuery) {
                alert('Please enter a search query');
                return;
            }
            
            const topK = document.getElementById('resultsPerPage').value;
            
            const resultsContainer = document.getElementById('resultsContainer');
            resultsContainer.innerHTML = `
                <div class="text-center py-5">
                    <div class="spinner-border text-primary" style="width: 3rem; height: 3rem;" role="status">
                        <span class="visually-hidden">Loading...</span>
                    </div>
                    <p class="mt-3 h5">Searching ${currentSearchType} for: <strong>"${currentQuery}"</strong></p>
                    <p class="text-muted">Please wait while we process your query...</p>
                </div>
            `;
            
            document.getElementById('resultsSection').style.display = 'block';
            document.getElementById('queryTime').innerHTML = '<span class="text-primary">Searching...</span>';
            
            try {
                const response = await fetch('/api/search', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json'
                    },
                    body: JSON.stringify({
                        query: currentQuery,
                        type: currentSearchType,
                        top_k: parseInt(topK)
                    })
                });
                
                const data = await response.json();
                
                if (data.error) {
                    throw new Error(data.error);
                }
                
                displayResults(data);
                
            } catch (error) {
                resultsContainer.innerHTML = `
                    <div class="alert alert-danger">
                        <i class="fas fa-exclamation-triangle me-2"></i>
                        <strong>Error:</strong> ${error.message || 'An unexpected error occurred'}
                    </div>
                `;
                document.getElementById('queryTime').innerHTML = '<span class="text-danger">Error</span>';
            }
        }

        function displayResults(data) {
            const resultsContainer = document.getElementById('resultsContainer');
            const resultsCount = document.getElementById('resultsCount');
            const queryTime = document.getElementById('queryTime');
            const pdfResultsCount = document.getElementById('pdfResultsCount');
            
            resultsCount.textContent = data.total_results;
            queryTime.innerHTML = `<span class="text-success">${data.query_time.toFixed(3)} seconds</span>`;
            
            // Count PDFs available
            const pdfCount = data.results.filter(r => r.has_pdf).length;
            pdfResultsCount.textContent = `${pdfCount} PDFs available`;
            
            if (data.total_results === 0) {
                resultsContainer.innerHTML = `
                    <div class="alert alert-warning">
                        <div class="d-flex align-items-center">
                            <i class="fas fa-search fa-2x me-3"></i>
                            <div>
                                <h5 class="alert-heading">No documents found</h5>
                                <p class="mb-0">No documents match your query: <strong>"${data.query}"</strong></p>
                            </div>
                        </div>
                    </div>
                `;
                return;
            }
            
            let html = '';
            
            data.results.forEach((result, index) => {
                const badgeClass = result.type.includes('boolean') ? 'bg-primary' : 
                                 result.type.includes('semantic') ? 'bg-success' : 
                                 result.type.includes('phrase') ? 'bg-info' : 'bg-warning';
                
                const badgeText = result.type.includes('boolean') ? 'Boolean' : 
                                result.type.includes('semantic') ? 'Semantic' : 
                                result.type.includes('phrase') ? 'Phrase' : 'Wildcard';
                
                const scorePercent = Math.min(Math.round(result.score * 100), 99);
                
                html += `
                    <div class="card result-card">
                        <div class="card-body">
                            <div class="d-flex justify-content-between align-items-start mb-2">
                                <div>
                                    <h5 class="card-title mb-1">
                                        ${result.title || result.doc_id}
                                        <span class="badge ${badgeClass} badge-search ms-2">${badgeText}</span>
                                        ${result.has_pdf ? '<span class="badge pdf-badge ms-2">PDF Available</span>' : ''}
                                    </h5>
                                    <h6 class="card-subtitle text-muted mb-2">
                                        <i class="fas fa-file-alt me-1"></i> ${result.filename || result.doc_id}
                                    </h6>
                                </div>
                                <div class="text-end">
                                    <div class="mb-2">
                                        <span class="badge bg-light text-dark">
                                            <i class="fas fa-ruler-combined me-1"></i> ${result.word_count ? result.word_count.toLocaleString() : 'N/A'} words
                                        </span>
                                    </div>
                                    <div class="d-flex align-items-center">
                                        <div class="progress" style="width: 80px; height: 8px;">
                                            <div class="progress-bar ${badgeClass}" role="progressbar" 
                                                 style="width: ${scorePercent}%" 
                                                 aria-valuenow="${scorePercent}" 
                                                 aria-valuemin="0" 
                                                 aria-valuemax="100">
                                            </div>
                                        </div>
                                        <small class="ms-2 text-muted">${result.score.toFixed(3)}</small>
                                    </div>
                                </div>
                            </div>
                            
                            <div class="row mt-3">
                                <div class="col-md-6">
                                    <p class="card-text text-muted">
                                        <i class="fas fa-info-circle me-1"></i>
                                        Document ID: <code>${result.doc_id}</code>
                                    </p>
                                </div>
                                <div class="col-md-6 text-end">
                                    <button class="btn btn-sm btn-outline-primary me-2" onclick="viewDocument('${result.doc_id}')">
                                        <i class="fas fa-eye me-1"></i> Preview
                                    </button>
                                    ${result.has_pdf ? `
                                        <a href="/download/pdf/${result.doc_id}" class="btn btn-download btn-sm me-2" target="_blank">
                                            <i class="fas fa-download me-1"></i> Download PDF
                                        </a>
                                    ` : `
                                        <button class="btn btn-sm btn-outline-secondary me-2" disabled title="PDF not available">
                                            <i class="fas fa-download me-1"></i> No PDF
                                        </button>
                                    `}
                                    <button class="btn btn-sm btn-outline-success" onclick="findSimilar('${result.doc_id}')">
                                        <i class="fas fa-project-diagram me-1"></i> Similar
                                    </button>
                                </div>
                            </div>
                        </div>
                    </div>
                `;
            });
            
            resultsContainer.innerHTML = html;
            document.getElementById('resultsSection').scrollIntoView({ behavior: 'smooth' });
        }

        async function viewDocument(docId) {
            try {
                const response = await fetch(`/api/document/${docId}`);
                const data = await response.json();
                
                if (data.error) {
                    throw new Error(data.error);
                }
                
                document.getElementById('documentTitle').innerHTML = `
                    <i class="fas fa-file-alt me-2"></i>${data.metadata.title || docId}
                `;
                document.getElementById('documentFilename').textContent = data.metadata.filename || docId;
                document.getElementById('documentLength').textContent = `${data.metadata.length?.toLocaleString() || 'N/A'} characters`;
                document.getElementById('documentWords').textContent = `${data.metadata.word_count?.toLocaleString() || 'N/A'} words`;
                document.getElementById('documentSentences').textContent = `${data.metadata.sentence_count || 'N/A'} sentences`;
                document.getElementById('documentContent').textContent = data.content;
                
                // Show/hide PDF download button in modal
                const pdfDownloadDiv = document.getElementById('pdfDownloadModal');
                const downloadLink = document.getElementById('modalDownloadLink');
                
                if (data.metadata.has_pdf) {
                    pdfDownloadDiv.style.display = 'block';
                    downloadLink.href = `/download/pdf/${docId}`;
                } else {
                    pdfDownloadDiv.style.display = 'none';
                }
                
                await loadSimilarDocuments(docId);
                
                const modal = new bootstrap.Modal(document.getElementById('documentModal'));
                modal.show();
                
            } catch (error) {
                alert('Error loading document: ' + error.message);
            }
        }

        async function findSimilar(docId) {
            try {
                const response = await fetch(`/api/similar/${docId}?top_k=5`);
                const data = await response.json();
                
                if (data.error) {
                    throw new Error(data.error);
                }
                
                document.getElementById('semanticQuery').value = `Similar to: ${docId}`;
                setExample('semantic', `Similar to: ${docId}`);
                
            } catch (error) {
                alert('Error finding similar documents: ' + error.message);
            }
        }

        async function loadSimilarDocuments(docId) {
            try {
                const response = await fetch(`/api/similar/${docId}?top_k=3`);
                const data = await response.json();
                
                const similarContainer = document.getElementById('similarDocuments');
                
                if (data.error || !data.similar_documents || data.similar_documents.length === 0) {
                    similarContainer.innerHTML = `
                        <div class="alert alert-info">
                            <i class="fas fa-info-circle me-2"></i>
                            No similar documents found.
                        </div>
                    `;
                    return;
                }
                
                let html = '<div class="list-group">';
                data.similar_documents.forEach((doc, index) => {
                    const similarityPercent = Math.round(doc.score * 100);
                    html += `
                        <a href="#" class="list-group-item list-group-item-action" 
                           onclick="viewDocument('${doc.doc_id}'); return false;">
                            <div class="d-flex w-100 justify-content-between align-items-center">
                                <div>
                                    <h6 class="mb-1">${doc.title || doc.doc_id}</h6>
                                    <small class="text-muted">${doc.filename || ''}</small>
                                </div>
                                <div class="text-end">
                                    <span class="badge bg-success">${similarityPercent}% match</span>
                                    <br>
                                    <small class="text-muted">${doc.word_count?.toLocaleString() || ''} words</small>
                                </div>
                            </div>
                        </a>
                    `;
                });
                html += '</div>';
                
                similarContainer.innerHTML = html;
            } catch (error) {
                similarContainer.innerHTML = `
                    <div class="alert alert-warning">
                        <i class="fas fa-exclamation-triangle me-2"></i>
                        Error loading similar documents.
                    </div>
                `;
            }
        }

        function clearResults() {
            document.getElementById('resultsSection').style.display = 'none';
            document.getElementById('resultsContainer').innerHTML = '';
            document.getElementById('resultsCount').textContent = '0';
            document.getElementById('pdfResultsCount').textContent = '0 PDFs available';
            document.getElementById('queryTime').innerHTML = 'Ready to search';
        }

        document.addEventListener('keypress', function(event) {
            if (event.key === 'Enter') {
                const activeInput = document.querySelector('.tab-pane.active input[type="text"]');
                if (activeInput && activeInput === document.activeElement) {
                    performSearch();
                }
            }
        });

        window.onload = function() {
            setExample('boolean', 'law AND order');
        };
    </script>
</body>
</html>
'''

# ========================= FLASK APP =========================
app = Flask(__name__)

# Store IR system globally
global_ir_system = None

@app.route('/')
def index():
    """Main page with search interface"""
    if global_ir_system:
        stats = global_ir_system.get_document_stats()
    else:
        stats = {'total_documents': 0, 'indexed_terms': 0, 'average_words_per_document': 0, 
                 'pdf_available': 0, 'top_terms': []}
    
    return render_template_string(HTML_TEMPLATE, stats=stats)

@app.route('/api/search', methods=['POST'])
def api_search():
    """API endpoint for search"""
    data = request.json
    query = data.get('query', '')
    search_type = data.get('type', 'boolean')
    top_k = data.get('top_k', 10)
    
    if not query or not global_ir_system:
        return jsonify({'error': 'System not ready or no query provided'}), 400
    
    try:
        result = global_ir_system.hybrid_search(query, search_type, top_k)
        return jsonify(result)
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/document/<doc_id>')
def api_document(doc_id):
    """Get document content"""
    if not global_ir_system:
        return jsonify({'error': 'System not ready'}), 400
    
    content = global_ir_system.get_document_content(doc_id, max_chars=10000)
    if content:
        metadata = global_ir_system.doc_metadata.get(doc_id, {})
        return jsonify({
            'doc_id': doc_id,
            'content': content,
            'metadata': metadata
        })
    
    return jsonify({'error': 'Document not found'}), 404

@app.route('/api/similar/<doc_id>')
def api_similar(doc_id):
    """Get similar documents"""
    if not global_ir_system:
        return jsonify({'error': 'System not ready'}), 400
    
    top_k = request.args.get('top_k', 5, type=int)
    similar = global_ir_system.get_similar_documents(doc_id, top_k)
    
    return jsonify({'similar_documents': similar})

@app.route('/api/stats')
def api_stats():
    """Get system statistics"""
    if global_ir_system:
        stats = global_ir_system.get_document_stats()
        return jsonify(stats)
    return jsonify({'error': 'System not ready'}), 400

@app.route('/download/pdf/<doc_id>')
def download_pdf(doc_id):
    """Download PDF file for a document"""
    if not global_ir_system:
        return jsonify({'error': 'System not ready'}), 400
    
    pdf_path = global_ir_system.get_pdf_path(doc_id)
    
    if not pdf_path or not os.path.exists(pdf_path):
        return jsonify({'error': 'PDF file not found'}), 404
    
    try:
        # Get the filename
        filename = os.path.basename(pdf_path)
        
        # Send the file
        return send_file(
            pdf_path,
            as_attachment=True,
            download_name=filename,
            mimetype='application/pdf'
        )
    except Exception as e:
        return jsonify({'error': f'Error downloading PDF: {str(e)}'}), 500

def start_flask_server(port=5000):
    """Start Flask server"""
    global global_ir_system
    
    print("\n" + "="*60)
    print("Initializing Supreme Court Judgement IR System")
    print("="*60)
    
    print(f"\nConfiguration:")
    print(f"PDF Directory: {PDF_DIR}")
    print(f"TXT Directory: {TXT_DIR}")
    print(f"Index Directory: {INDEX_DIR}")
    
    # Initialize the IR system
    global_ir_system = IRSystem(use_existing_indices=True)
    
    print(f"\nStarting web server on http://localhost:{port}")
    print("✓ System is ready!")
    print("✓ PDF Download feature enabled")
    print("\n" + "="*60)
    print("Open your browser and go to: http://localhost:5000")
    print("="*60)
    
    # Run Flask
    app.run(host='0.0.0.0', port=port, debug=False, use_reloader=False)

# Start the Flask server
if __name__ == "__main__":
    start_flask_server(port=5000)

✓ All packages installed and ready!
PDF Directory: C:\Users\Armaghan Rafique\Desktop\AI Project\supreme_court_judgements_pdf
TXT Directory: C:\Users\Armaghan Rafique\Desktop\AI Project\supreme_court_judgements_txt
Index Directory: indices

Initializing Supreme Court Judgement IR System

Configuration:
PDF Directory: C:\Users\Armaghan Rafique\Desktop\AI Project\supreme_court_judgements_pdf
TXT Directory: C:\Users\Armaghan Rafique\Desktop\AI Project\supreme_court_judgements_txt
Index Directory: indices

Loading documents from text files...
Found 1474 text files in C:\Users\Armaghan Rafique\Desktop\AI Project\supreme_court_judgements_txt
Processing ALL 1474 text files...
  Loaded 100/1474 files (100 valid, 0 skipped)
  Loaded 200/1474 files (200 valid, 0 skipped)
  Loaded 300/1474 files (300 valid, 0 skipped)
  Loaded 400/1474 files (400 valid, 0 skipped)
  Loaded 500/1474 files (500 valid, 0 skipped)
  Loaded 600/1474 files (600 valid, 0 skipped)
  Loaded 700/1474 files (700 valid, 0 ski

 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.26.5.79:5000
Press CTRL+C to quit
172.26.5.79 - - [30/Dec/2025 16:53:19] "GET / HTTP/1.1" 200 -
172.26.5.79 - - [30/Dec/2025 16:53:29] "POST /api/search HTTP/1.1" 200 -
